In [1]:
import pyzx as zx

c = zx.Circuit(2)
c.add_gate("HAD", 0)
c.add_gate("CNOT", 0, 1)

g = c.to_graph()
zx.draw(g)          # diagrama ZX del circuito
zx.full_reduce(g)   # aplica las reglas de reescritura
zx.draw(g)          # diagrama simplificado

In [3]:
circuit = zx.Circuit(2)
circuit.add_gate("T", 0)
circuit.add_gate("CNOT", 0, 1)
g = circuit.to_graph()
zx.draw(g)  

In [11]:
circuit = zx.Circuit(2)
#circuit.add_gate("T",0)
circuit.add_gate("CNOT",0,1 )
graph = circuit.to_graph()
graph.apply_state("0")
zx.draw(graph)

Los nodos de un mismo color se fusionan. Porque es como aplicar la identidad y sale la misma salida

In [12]:
import pyzx as zx

c = zx.Circuit(2)
c.add_gate("HAD", 0)
c.add_gate("HAD", 0)              # H·H = I
c.add_gate("CNOT", 0, 1)
c.add_gate("CNOT", 0, 1)          # CNOT·CNOT = I
c.add_gate("T", 1)
c.add_gate("T", 1, adjoint=True)  # T·T† = I

g = c.to_graph()
zx.draw(g)          # antes: varias arañas y cajas
zx.full_reduce(g)
zx.draw(g)          # después: dos cables rectos

In [10]:
import pyzx as zx
from fractions import Fraction

c = zx.Circuit(1)
c.add_gate("XPhase", 0, phase=Fraction(1, 4))   # roja con fase π/4
c.add_gate("XPhase", 0, phase=Fraction(1, 2))   # roja con fase π/2

g = c.to_graph()
zx.draw(g)                    # dos nodos rojos

zx.simplify.spider_simp(g)    # solo fusión de arañas
zx.draw(g)                    # un nodo rojo con fase 3π/4

In [13]:
import pyzx as zx

c = zx.Circuit(2)
c.add_gate("HAD", 0)
c.add_gate("HAD", 0)              # H·H = I


g = c.to_graph()
zx.draw(g)          # antes: varias arañas y cajas
zx.full_reduce(g)
zx.draw(g)          # después: dos cables rectos

In [14]:
c = zx.Circuit(1)
c.add_gate("HAD", 0)
c.add_gate("HAD", 0)

g = c.to_graph()
zx.full_reduce(g)
zx.draw(g)

In [ ]:
g = c.to_graph()
g.apply_state("0")
zx.full_reduce(g)
zx.draw(g)
print(g.to_tensor())   # ≈ [1, 0]  → |0⟩

[1.+0.j 0.+0.j]


In [16]:
g = c.to_graph()
g.apply_state("0")
zx.simplify.spider_simp(g)
zx.simplify.id_simp(g)
zx.draw(g)

In [17]:
n = 4
c = zx.Circuit(n)
for q in range(n): c.add_gate("HAD", q)
for q in range(n): c.add_gate("HAD", q)

g = c.to_graph()
zx.full_reduce(g)
zx.draw(g)                                   # n cables rectos
print(zx.compare_tensors(c, zx.Circuit(n)))  # True

True


In [18]:
import pyzx as zx
from fractions import Fraction
from pyzx import VertexType, EdgeType

def arana(color, h_entrada, h_salida, fase=Fraction(1, 4)):
    g = zx.Graph()
    ins  = [g.add_vertex(VertexType.BOUNDARY, qubit=q, row=0) for q in range(2)]
    outs = [g.add_vertex(VertexType.BOUNDARY, qubit=q, row=4) for q in range(2)]
    c = g.add_vertex(color, qubit=0.5, row=2, phase=fase)
    for i in ins:
        g.add_edge((i, c), EdgeType.HADAMARD if h_entrada else EdgeType.SIMPLE)
    for o in outs:
        g.add_edge((c, o), EdgeType.HADAMARD if h_salida else EdgeType.SIMPLE)
    g.set_inputs(ins)
    g.set_outputs(outs)
    return g, c
izq, v = arana(VertexType.Z, h_entrada=True,  h_salida=False)  # verde, H a la entrada
der, _ = arana(VertexType.X, h_entrada=False, h_salida=True)   # roja, H a la salida

zx.draw(izq)
zx.draw(der)
print(zx.compare_tensors(izq, der))   # True

todas_h, _ = arana(VertexType.Z, h_entrada=True,  h_salida=True)
roja, _    = arana(VertexType.X, h_entrada=False, h_salida=False)
print(zx.compare_tensors(todas_h, roja))   # True

True
True


In [21]:
(print(zx.__version__))

0.10.6


In [22]:
from pyzx.rewrite_rules.color_change_rule import color_change
g = izq.copy()
color_change(g, v)   # cambia el color del nodo e invierte el tipo de cada arista
zx.draw(g)           # roja, con las Hadamards ahora en la salida